In [1]:
%matplotlib inline
# IPython uses asyncio, as does resample_image()
import nest_asyncio
nest_asyncio.apply()
# Utilities
from functools import partial
from hashlib import sha256
from pathlib import Path
from pprint import pprint

def print_files(*files):
    if len(files) == 1 and isinstance(files[0], list):
        files = files[0]
    pprint([Path(file).name for file in files])

def checksum(image):
    return sha256(image.to_bytes()).hexdigest()[:8]

# Resampling BOLD data in-process

This notebook walks through the procedure for resampling raw BOLD data using fMRIPrep's minimal derivatives. The reader will be load a BOLD file and a target template and then, using the transforms found in the fMRIPrep derivatives, resample the file directly to the template, applying head-motion correction, susceptibility distortion correction in the process.

We will be taking advantage of these Python libraries for data fetching, loading and transformations:

In [2]:
import nibabel as nb
import nitransforms as nt
from bids import BIDSLayout
from nilearn import image as nli
from templateflow import TemplateFlowClient

tf = TemplateFlowClient()

We will also use these tools from Nipreps packages:

In [3]:
from fmriprep.interfaces.resampling import resample_image, reconstruct_fieldmap
from fmriprep.utils.bids import collect_derivatives as collect_func, collect_fieldmaps
from fmriprep.utils.transforms import load_transforms
from sdcflows.utils.epimanip import get_trt
from sdcflows.utils.tools import ensure_positive_cosines
from smriprep.utils.bids import collect_derivatives as collect_anat

We begin by loading the raw and derivative datasets using PyBIDS' `BIDSLayout`. (Set `validate=False` for derivatives, because we want to load files that are not yet standardized in BIDS.)

In [4]:
raw = BIDSLayout('inputs/ds005365')
deriv = BIDSLayout('inputs/ds005365-fmriprep/', validate=False)

Let's retrieve the first BOLD file we can find, and target the `MNI152NLin2009cAsym` template space:

In [5]:
template = 'MNI152NLin2009cAsym'

print_files(bold_file := raw.get(suffix='bold', extension='.nii.gz')[0])
print_files(MNI_file := tf.get(template=template, suffix='mask', desc='brain', resolution='02'))

['sub-CISC13877_task-rest_dir-AP_run-01_bold.nii.gz']
['tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz']


Now retrieve the BIDS *entities* that identify the BOLD file. We will use these to identify relevant derivatives.

In [6]:
bold_entities = bold_file.get_entities()
for ent in ('datatype', 'suffix', 'extension'):
    bold_entities.pop(ent, None)
print(bold_entities)
subject = bold_entities['subject']

{'subject': 'CISC13877', 'task': 'rest', 'direction': 'AP', 'run': 01}


fMRIPrep (and sMRIPrep) have utilities for collecting pre-computed derivatives, and we will take advantage of those, to avoid writing our own custom queries. These are wrappers around the `deriv.get()` method that return dictionaries.

In [7]:
anat_derivs = collect_anat(deriv.root, subject_id=subject, std_spaces=[template])
func_derivs = collect_func(deriv.root, entities=bold_entities)
fmap_derivs = collect_fieldmaps(deriv.root, entities={'subject': subject})

Now we collect the spatial transforms that map from each BOLD volume to MNI:

In [8]:
hmc_xfm = func_derivs['transforms']['hmc']
boldref2anat_xfm = func_derivs['transforms']['boldref2anat']
anat2std_xfm = anat_derivs['transforms']['MNI152NLin2009cAsym']['forward']
bold2std_xfms = [hmc_xfm, boldref2anat_xfm, anat2std_xfm]

print_files(bold2std_xfms)

['sub-CISC13877_task-rest_dir-AP_run-01_from-orig_to-boldref_mode-image_desc-hmc_xfm.txt',
 'sub-CISC13877_task-rest_dir-AP_run-01_from-boldref_to-T1w_mode-image_desc-coreg_xfm.txt',
 'sub-CISC13877_from-T1w_to-MNI152NLin2009cAsym_mode-image_xfm.h5']


fMRIPrep will have created registrations from each BOLD file to its applicable fieldmap(s). Find these and create a transform chain to the target space:

In [9]:
boldref2fmap_xfm = func_derivs['transforms']['boldref2fmap'][0]
fmap2std_xfms = [boldref2fmap_xfm, boldref2anat_xfm, anat2std_xfm]
fmap2std_inv = [True, False, False]

print_files(fmap2std_xfms)

['sub-CISC13877_task-rest_dir-AP_run-01_from-boldref_to-auto00000_mode-image_xfm.txt',
 'sub-CISC13877_task-rest_dir-AP_run-01_from-boldref_to-T1w_mode-image_desc-coreg_xfm.txt',
 'sub-CISC13877_from-T1w_to-MNI152NLin2009cAsym_mode-image_xfm.h5']


To find the actual fieldmap, we will extract the fieldmap ID from the `to-` entity in the transform. Then we will load the fieldmap reference image and the coefficients that describe the field inhomogeneity:

In [10]:
# Extract fieldmap ID and find fieldmap
fmapid = deriv.files[boldref2fmap_xfm].entities['to']
fmapref_file = fmap_derivs[fmapid]['magnitude']
print_files(coeff_file := fmap_derivs[fmapid]['coeffs'])

['sub-CISC13877_run-01_fmapid-auto00000_desc-coeff_fieldmap.nii.gz']


We now have all of the files we need, but we need the phase-encoding direction and total readout time of the BOLD image to perform fieldmap correction. The metadata needs to be in a particular format, so we'll write a function to retrieve these:

In [11]:
def prepare_bold(bold_file: bids.layout.BIDSFile) -> tuple[nb.Nifti1Image, list[tuple[int, float]]]:
    bold = nb.load(bold_file)
    # The math is simpler if all directions are positive (e.g., RAS instead of LAS)
    # Reorient the BOLD data and record the original orientation
    source, axcodes = ensure_positive_cosines(bold)

    metadata = bold_file.get_metadata()
    trt = get_trt(metadata, bold_file)
    pe_dir = metadata['PhaseEncodingDirection']
    pe_axis = 'ijk'.index(pe_dir[0])

    # Deflection is negative if the PE dir is negative XOR we flipped the PE axis
    pe_flip = pe_dir.endswith('-')
    axis_flip = axcodes[pe_axis] in 'LPI'

    # Record the axis and the signed readout time for each volume
    pe_info = [(pe_axis, -trt if (axis_flip ^ pe_flip) else trt)] * source.shape[3]

    return source, pe_info

Now we are ready to load our images. Use the function above to load and reorient the BOLD image and prepare its phase-encoding information. For the sake of speed, we will crop the MNI reference to the brain mask.

Additionally, we will load the transforms as chains.

In [12]:
bold, pe_info = prepare_bold(bold_file)
MNI = nli.crop_img(MNI_file)
fmapref = nb.load(fmapref_file)
coeff = nb.load(coeff_file)

bold2std = load_transforms(bold2std_xfms, inverse=[False])
fmap2std = load_transforms(fmap2std_xfms, inverse=fmap2std_inv)

The fieldmap needs to be reconstructed in the target space (see [Rethinking resampling](https://hackmd.io/@effigies/rethinking-resampling) for a full explanation):

In [13]:
fmap_std = reconstruct_fieldmap([coeff], fmapref, MNI, fmap2std)

And finally, resample the BOLD image to MNI using fMRIPrep's `resample_image` function:

bold_mni = resample_image(
    source=bold,
    target=MNI,
    transforms=bold2std,
    fieldmap=fmap_std, 
    pe_info=pe_info,
    jacobian=True,
    mode='grid-constant',
)

This takes about 2 minutes on my laptop. Once complete, we can view the image:

In [14]:
bold_mni.slicer[..., :10].orthoview();

NameError: name 'bold_mni' is not defined

You can also verify that you got the same result as I did by comparing the checksum:

In [ ]:
checksum(bold_mni) == 'e822b06c'

That concludes this tutorial. You have now resampled a BOLD image to MNI in-memory. You can now save the image (`bold_mni.to_filename(...)`), or perhaps use [Nilearn](https://nilearn.github.io/) to perform a statistical analysis!